# 14 · Silver: pontos de descarte (datasets 3 + 4 + 5 + lista de apoio)

Junta os três cadastros públicos e a lista de apoio das 17 ecoestações numa entidade só, unindo registros a até 50 m.

**Regra de honestidade:** nenhuma linha confirma funcionamento atual. Toda situação nasce *histórico, a confirmar*, com data de verificação *desconhecida*. As 7 ecoestações novas da lista de apoio ficam com **localização pendente** (sem coordenada conferida). Horário não é inventado.

**Papel no Canvas:** problema 1 (destinação inadequada). App Cidadão (*Onde descartar*) e painel; primeiro fluxo.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "src" / "wastestack").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))
import pandas as pd
from wastestack.arquivos import ler
from wastestack.config import SILVER, GOLD

In [2]:
from wastestack.silver import preparar_descarte
resumo = preparar_descarte()
resumo

{
 "linhas_fonte": 332,
 "linhas_fora_do_recife": 19,
 "locais": 184,
 "locais_com_coordenada": 177,
 "por_tipo": {
  "PEV": 51,
  "não informado": 47,
  "parceiro ou outro": 34,
  "ecoestação": 20,
  "ecoponto": 17,
  "cooperativa": 15
 },
 "ecoestacoes": 20,
 "ecoestacoes_localizacao_pendente": 7,
 "bairros_sem_local": 34,
 "locais_que_recebem_entulho": 20
}


In [3]:
pd_ = ler("ponto_descarte")
assert pd_.ponto_id.is_unique and (pd_.situacao == "histórico, a confirmar").all()
assert resumo["ecoestacoes_localizacao_pendente"] == 7
pd_.groupby("tipo").agg(locais=("ponto_id", "size"), com_coordenada=("localizacao", lambda s: (s != "pendente").sum()))

                   locais  com_coordenada
tipo                                     
PEV                    51              51
cooperativa            15              15
ecoestação             20              13
ecoponto               17              17
não informado          47              47
parceiro ou outro      34              34


## Limites e cuidados

- As ecoestações aparecem em mais de um endereço entre as fontes (5 divergências registradas na coluna `divergencia_registrada`); os registros a mais de 50 m ficam como locais separados até a vistoria de mesa.
- Os materiais vêm do texto `tiporesiduo` da fonte. As colunas SIM/NÃO do dataset 4 não são usadas (estão inconsistentes).
- 19 linhas ficam fora do domínio do Recife e saem da tabela.